In [11]:
from typing import Tuple
import torch
from torch import Tensor
import torch.nn as nn
import math
from scipy.optimize import linear_sum_assignment
import numpy as np

def round_to_perm(P):
    N = P.shape[0]
    assert P.shape == (N, N)
    row, col = linear_sum_assignment(-P)
    P = np.zeros((N, N))
    P[row, col] = 1.0
    return P


device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

def min_zero_row(zero_mat: Tensor) -> Tuple[Tensor, Tensor]:
    sum_zero_mat = zero_mat.sum(1)
    sum_zero_mat[sum_zero_mat == 0] = 9999

    zero_row = sum_zero_mat.min(0)[1]
    zero_column = zero_mat[zero_row].nonzero()[0]

    zero_mat[zero_row, :] = False
    zero_mat[:, zero_column] = False

    mark_zero = torch.tensor([[zero_row, zero_column]], device = device)
    return zero_mat, mark_zero

def mark_matrix(mat: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
    zero_bool_mat = (mat == 0)
    zero_bool_mat_copy = zero_bool_mat.clone()

    marked_zero = torch.tensor([], device = device)
    while (True in zero_bool_mat_copy):
        zero_bool_mat_copy, mark_zero = min_zero_row(zero_bool_mat_copy)
        marked_zero = torch.concat([marked_zero, mark_zero], dim = 0)

    marked_zero_row = marked_zero[:, 0]
    marked_zero_col = marked_zero[:, 1]

    arange_index_row = torch.arange(mat.shape[0], dtype=torch.float, device = device).unsqueeze(1)
    
    repeated_marked_row = marked_zero_row.repeat(mat.shape[0], 1)
    bool_non_marked_row = torch.all(arange_index_row != repeated_marked_row, dim = 1)
    non_marked_row = arange_index_row[bool_non_marked_row].squeeze()

    non_marked_mat = zero_bool_mat[non_marked_row.long(), :]
    marked_cols = non_marked_mat.nonzero().unique()

    is_need_add_row = True
    while is_need_add_row:
        repeated_non_marked_row = non_marked_row.repeat(marked_zero_row.shape[0], 1)
        repeated_marked_cols = marked_cols.repeat(marked_zero_col.shape[0], 1)

        first_bool = torch.all(marked_zero_row.unsqueeze(1) != repeated_non_marked_row, dim = 1)
        second_bool = torch.any(marked_zero_col.unsqueeze(1) == repeated_marked_cols, dim = 1)

        addit_non_marked_row = marked_zero_row[first_bool & second_bool]

        if addit_non_marked_row.shape[0] > 0:
            non_marked_row = torch.concat([non_marked_row.reshape(-1), addit_non_marked_row[0].reshape(-1)])
        else:
            is_need_add_row = False

    repeated_non_marked_row = non_marked_row.repeat(mat.shape[0], 1)
    bool_marked_row = torch.all(arange_index_row != repeated_non_marked_row, dim = 1)
    marked_rows = arange_index_row[bool_marked_row].squeeze(0)

    return marked_zero, marked_rows, marked_cols

def adjust_matrix(mat: Tensor, cover_rows: Tensor, cover_cols: Tensor) -> Tensor:
    bool_cover = torch.zeros_like(mat)
    bool_cover[cover_rows.long()] = True
    bool_cover[:, cover_cols.long()] = True

    non_cover = mat[bool_cover != True]
    min_non_cover = non_cover.min()

    mat[bool_cover != True] = mat[bool_cover != True] - min_non_cover

    double_bool_cover = torch.zeros_like(mat)
    double_bool_cover[cover_rows.long(), cover_cols.long()] = True

    mat[double_bool_cover == True] = mat[double_bool_cover == True] + min_non_cover

    return mat

def hungarian_algorithm(mat: Tensor) -> Tensor:
    dim = mat.shape[0]
    cur_mat = mat.clone()

    cur_mat = cur_mat - cur_mat.min(1, keepdim=True)[0]
    cur_mat = cur_mat - cur_mat.min(0, keepdim=True)[0]

    zero_count = 0
    iters = 0
    while zero_count < dim and iters < 100:
        ans_pos, marked_rows, marked_cols = mark_matrix(cur_mat)
        zero_count = len(marked_rows) + len(marked_cols)

        if zero_count < dim:
            cur_mat = adjust_matrix(cur_mat, marked_rows, marked_cols)
        iters += 1

    # Create permutation matrix
    perm_matrix = torch.zeros_like(mat)
    for pos in ans_pos:
        i, j = int(pos[0]), int(pos[1])  # Ensure indices are integers
        perm_matrix[i, j] = 1

    return perm_matrix

def sinkhorn_logspace(logP, niters=10):
    for _ in range(niters):
        # Normalize columns and take the log again
        logP = logP - torch.logsumexp(logP, dim=0, keepdim=True)
        # Normalize rows and take the log again
        logP = logP - torch.logsumexp(logP, dim=1, keepdim=True)
    return logP

In [12]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch
import torch.optim as optim
from tqdm import tqdm

# Add the path to the src directory
sys.path.append(os.path.abspath(os.path.join('..', 'data')))
from generation import generate_data, partition_domain_into_regions


In [13]:
# Import functions from generation.py
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from trainer_debug import trainer, compute_q_phi
from elbo import vi_piX, vi_piS

In [14]:
B = 1 # Number of regions
n_i = 50 # Number of locations per region

# Generate data
y, x, w, e, s, region_assignments = generate_data(B, n_i, sigmasq = 0.005, length_scale=0.5, nu=0.5, seed=50, beta_true=8, tausq_true=0.25,spatial=False)

# jumbled data generation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = 1
# num_neighbors= 3

y = torch.tensor(y, dtype=torch.float32, device=device)
x = torch.tensor(x, dtype=torch.float32, device=device)
w = torch.tensor(w, dtype=torch.float32, device=device)
e = torch.tensor(e, dtype=torch.float32, device=device)
s = torch.tensor(s, dtype=torch.float32, device=device)
region_assignments = torch.tensor(region_assignments, dtype=torch.int64, device=device)  # Region indices as integers



unique_regions = torch.unique(region_assignments)


# Jumble x and s within regions
x_jumbled_within_regions = torch.zeros_like(x)
s_jumbled_within_regions = torch.zeros_like(s)
torch.manual_seed(42)  # Set a seed for reproducibility
perm_x = torch.randperm(n_i)
perm_s = torch.randperm(n_i)

for i, region in enumerate(unique_regions):
    # Get indices for the current region
    indices = torch.where(region_assignments == region)[0]
    
    # Shuffle the indices within the region
    
    # Assign the shuffled values back
    x_jumbled_within_regions[indices] = x[indices[perm_x]]
    s_jumbled_within_regions[indices] = s[indices[perm_s]]

# Print the jumbled data
print("Jumbled x within regions:", x_jumbled_within_regions)
print("Jumbled s within regions:", s_jumbled_within_regions)

# Convert perm_s into a permutation matrix
perm_matrix_s = torch.zeros(n_i, n_i, dtype=torch.float32, device=device)
perm_matrix_s[torch.arange(n_i), perm_s] = 1

# Convert perm_x into a permutation matrix
perm_matrix_x = torch.zeros(n_i, n_i, dtype=torch.float32, device=device)
perm_matrix_x[torch.arange(n_i), perm_x] = 1

# Print the permutation matrices
print("Permutation matrix for perm_s:\n", perm_matrix_s)
print("Permutation matrix for perm_x:\n", perm_matrix_x)

Y= y.reshape(B, n_i)
X= x_jumbled_within_regions.reshape(B, n_i)

Jumbled x within regions: tensor([[0.4151],
        [0.8281],
        [0.4073],
        [0.1738],
        [0.5883],
        [0.1793],
        [0.0497],
        [0.2124],
        [0.9024],
        [0.1755],
        [0.9060],
        [0.4057],
        [0.4870],
        [0.4082],
        [0.3179],
        [0.4627],
        [0.5949],
        [0.9870],
        [0.5024],
        [0.3809],
        [0.5930],
        [0.8302],
        [0.6382],
        [0.2972],
        [0.5821],
        [0.4126],
        [0.4712],
        [0.7953],
        [0.1665],
        [0.0370],
        [0.0898],
        [0.7726],
        [0.0705],
        [0.7070],
        [0.0165],
        [0.9045],
        [0.2920],
        [0.8837],
        [0.6387],
        [0.2444],
        [0.7345],
        [0.0421],
        [0.3700],
        [0.4581],
        [0.2091],
        [0.1273],
        [0.9851],
        [0.2354],
        [0.3085],
        [0.9611]])
Jumbled s within regions: tensor([[0.6499, 0.8403],
        [0.5241, 0.96

In [15]:
# Test code to check ELBO calculation
# Initialize parameters
n_locations = n_i
batch_size = B
n_sample = 40

# Create dummy data
torch.manual_seed(50)  # Set seed for reproducibility
# Generate a random permutation matrix pi_X_true
pi_X_true = perm_matrix_x
mu_lambda_beta = 8
sigmasq_lambda_beta = 0.0032
M_S_star = perm_matrix_s.T
#mu_W = torch.randn(batch_size, n_locations)
mu_W = (M_S_star.T @ Y.T - mu_lambda_beta * M_S_star.T @ X.T).T
eta_X_sq = 0.1 
lambda_a2 = 2
lambda_b2 = 0.25
lambda_a1=2
lambda_b1 = 5
tau_X = 0.1

In [24]:
from typing import Tuple
import torch
from torch import Tensor
import torch.nn as nn
import math
from scipy.optimize import linear_sum_assignment
import numpy as np

def round_to_perm(P):
    N = P.shape[0]
    assert P.shape == (N, N)
    row, col = linear_sum_assignment(-P)
    P = np.zeros((N, N))
    P[row, col] = 1.0
    return P


# device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# def min_zero_row(zero_mat: Tensor) -> Tuple[Tensor, Tensor]:
#     sum_zero_mat = zero_mat.sum(1)
#     sum_zero_mat[sum_zero_mat == 0] = 9999

#     zero_row = sum_zero_mat.min(0)[1]
#     zero_column = zero_mat[zero_row].nonzero()[0]

#     zero_mat[zero_row, :] = False
#     zero_mat[:, zero_column] = False

#     mark_zero = torch.tensor([[zero_row, zero_column]], device = device)
#     return zero_mat, mark_zero

# def mark_matrix(mat: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
#     zero_bool_mat = (mat == 0)
#     zero_bool_mat_copy = zero_bool_mat.clone()

#     marked_zero = torch.tensor([], device = device)
#     while (True in zero_bool_mat_copy):
#         zero_bool_mat_copy, mark_zero = min_zero_row(zero_bool_mat_copy)
#         marked_zero = torch.concat([marked_zero, mark_zero], dim = 0)

#     marked_zero_row = marked_zero[:, 0]
#     marked_zero_col = marked_zero[:, 1]

#     arange_index_row = torch.arange(mat.shape[0], dtype=torch.float, device = device).unsqueeze(1)
    
#     repeated_marked_row = marked_zero_row.repeat(mat.shape[0], 1)
#     bool_non_marked_row = torch.all(arange_index_row != repeated_marked_row, dim = 1)
#     non_marked_row = arange_index_row[bool_non_marked_row].squeeze()

#     non_marked_mat = zero_bool_mat[non_marked_row.long(), :]
#     marked_cols = non_marked_mat.nonzero().unique()

#     is_need_add_row = True
#     while is_need_add_row:
#         repeated_non_marked_row = non_marked_row.repeat(marked_zero_row.shape[0], 1)
#         repeated_marked_cols = marked_cols.repeat(marked_zero_col.shape[0], 1)

#         first_bool = torch.all(marked_zero_row.unsqueeze(1) != repeated_non_marked_row, dim = 1)
#         second_bool = torch.any(marked_zero_col.unsqueeze(1) == repeated_marked_cols, dim = 1)

#         addit_non_marked_row = marked_zero_row[first_bool & second_bool]

#         if addit_non_marked_row.shape[0] > 0:
#             non_marked_row = torch.concat([non_marked_row.reshape(-1), addit_non_marked_row[0].reshape(-1)])
#         else:
#             is_need_add_row = False

#     repeated_non_marked_row = non_marked_row.repeat(mat.shape[0], 1)
#     bool_marked_row = torch.all(arange_index_row != repeated_non_marked_row, dim = 1)
#     marked_rows = arange_index_row[bool_marked_row].squeeze(0)

#     return marked_zero, marked_rows, marked_cols

# def adjust_matrix(mat: Tensor, cover_rows: Tensor, cover_cols: Tensor) -> Tensor:
#     bool_cover = torch.zeros_like(mat)
#     bool_cover[cover_rows.long()] = True
#     bool_cover[:, cover_cols.long()] = True

#     non_cover = mat[bool_cover != True]
#     min_non_cover = non_cover.min()

#     mat[bool_cover != True] = mat[bool_cover != True] - min_non_cover

#     double_bool_cover = torch.zeros_like(mat)
#     double_bool_cover[cover_rows.long(), cover_cols.long()] = True

#     mat[double_bool_cover == True] = mat[double_bool_cover == True] + min_non_cover

#     return mat

# def hungarian_algorithm(mat: Tensor) -> Tensor:
#     dim = mat.shape[0]
#     cur_mat = mat.clone()

#     cur_mat = cur_mat - cur_mat.min(1, keepdim=True)[0]
#     cur_mat = cur_mat - cur_mat.min(0, keepdim=True)[0]

#     zero_count = 0
#     iters = 0
#     while zero_count < dim and iters < 100:
#         ans_pos, marked_rows, marked_cols = mark_matrix(cur_mat)
#         zero_count = len(marked_rows) + len(marked_cols)

#         if zero_count < dim:
#             cur_mat = adjust_matrix(cur_mat, marked_rows, marked_cols)
#         iters += 1

#     # Create permutation matrix
#     perm_matrix = torch.zeros_like(mat)
#     for pos in ans_pos:
#         i, j = int(pos[0]), int(pos[1])  # Ensure indices are integers
#         perm_matrix[i, j] = 1

#     return perm_matrix

def sinkhorn_logspace(logP, niters=10):
    for _ in range(niters):
        # Normalize columns and take the log again
        logP = logP - torch.logsumexp(logP, dim=0, keepdim=True)
        # Normalize rows and take the log again
        logP = logP - torch.logsumexp(logP, dim=1, keepdim=True)
    return logP

class vi_piX(nn.Module):
    def __init__(self, n_locations):
        super(vi_piX, self).__init__()
        self.n_locations = n_locations
        random_matrix = torch.rand(n_locations, n_locations)
        log_doubly_stochastic = sinkhorn_logspace((random_matrix), niters=10)
        self.MX = nn.Parameter(log_doubly_stochastic, requires_grad=True)
        self.VX = nn.Parameter((-2)*torch.ones(n_locations, n_locations, requires_grad=True))
        self.current_M_X_star = (1/torch.tensor(n_locations)) * torch.ones(n_locations, n_locations)
        self.current_V_X_star = torch.eye(n_locations, n_locations)
        #self.VX_unconstrained = nn.Parameter(torch.full((n_locations, n_locations), 0.2))
            
    def forward(self, Y, X, mu_lambda_beta,
                sigmasq_lambda_beta, M_S_star, mu_W,
                eta_X_sq, lambda_a2, lambda_b2, 
                tau_X = 0.1, n_piX_sample = 100, seed = 100):
        
        # Enable anomaly detection
        torch.autograd.set_detect_anomaly(True)

        # sample piX
        torch.manual_seed(seed=seed)  #set seed for stochastic optimzation

        # calculate nearest doubly stochastic matrix to MX once
        log_MX = self.MX
        log_MX_tilde = sinkhorn_logspace(log_MX, niters=10)
        MX_tilde = torch.exp(log_MX_tilde)
        #VX = torch.nn.functional.softplus(self.VX_unconstrained)
    

        # Compute the ELBO
        B = Y.shape[0]
        elbo = 0.0
        current_M_X_star = self.current_M_X_star
        current_V_X_star = self.current_V_X_star
        for i in range(n_piX_sample):
            #Phi = MX_tilde + torch.sqrt(torch.exp(self.VX)) * torch.randn(self.n_locations, self.n_locations)
            Phi = MX_tilde + torch.sqrt(torch.special.expit(self.VX)*(0.25-0.01) + 0.01) * torch.randn(self.n_locations, self.n_locations)
            round_Phi = round_to_perm((Phi).detach().numpy())
            current_piX = tau_X * Phi + (1 - tau_X) * torch.tensor(round_Phi, dtype=Phi.dtype)  # Access the current sample of piS
            #current_piX = tau_X * Phi + (1 - tau_X) * hungarian_algorithm(-Phi)
            #current_piX = tau_X * Phi + (1 - tau_X) * torch.tensor(round_to_perm((Phi - 0.95*Phi.min()).detach().numpy()), dtype=torch.float) # Access the current sample of piX
            current_M_X_star += current_piX
            current_V_X_star += current_piX.T @ current_piX

            # Computation for all regions
            term1 = 0.
            for j in range(B):
                Y_i = Y[j]                    # (d,)
                X_i = X[j]                    # (d,)
                mu_Wi = mu_W[j]              # (d,)

                # (1) - 2 * Y_i^T * pi_x * X_i * mu
                temp = current_piX @ X_i 
                part1 = -2 * mu_lambda_beta * torch.dot(Y_i, temp)

                # (2) (mu^2 + sigma^2) * X_i^T * pi_x^T * pi_x * X_i
                part2 = (mu_lambda_beta ** 2 + sigmasq_lambda_beta) * temp.dot(temp)



                # (3) 2 * mu * X_i^T * pi_x^T * M_star_S * mu_Wi
                part3 = 2 * mu_lambda_beta * (temp.T @ M_S_star @ mu_Wi)

                term1 += part1 + part2 + part3

            coeff = -lambda_a2 / (2 * lambda_b2)
            total_term1 = coeff * term1

            # Second summation: over entries of pi_x
            x_mk_squared = current_piX.pow(2)
            x_mk_minus1_squared = (current_piX - 1).pow(2)

            exponent1 = -x_mk_squared / (2 * eta_X_sq)
            exponent2 = -x_mk_minus1_squared / (2 * eta_X_sq)

            # Stable log-sum-exp
            log_term = torch.logsumexp(torch.stack([exponent1, exponent2]), dim=0)
            total_log_term = (log_term).sum()

            # Update ELBO
            #elbo += total_term1 + total_log_term + neg_log_tauX + 0.5 * self.VX.sum()
            elbo += total_term1 + total_log_term # Ensure positive definiteness
            #elbo += total_term1 + neg_log_tauX + 2 * self.VX.sum()

        
        elbo = elbo / n_piX_sample + self.n_locations ** 2 * torch.log(torch.tensor(tau_X)) + 0.5* torch.log(torch.special.expit(self.VX)*(0.25-0.01) + 0.01).sum()
        self.current_M_X_star = current_M_X_star / n_piX_sample
        self.current_V_X_star = current_V_X_star / n_piX_sample
        return -elbo



In [25]:
import seaborn as sns

import torch.optim as optim

# Initialize the model
model = vi_piX(n_locations=n_locations)

# Define the optimizer
optimizer = optim.AdamW(model.parameters(), lr=0.1)

# Number of optimization steps
n_steps = 150
    
        
import matplotlib.pyplot as plt

# Store losses for plotting
losses = []

# Store the permutation matrix error for plotting
perm_matrix_errors = []

# Update the loop to collect data
for step in range(n_steps):
    optimizer.zero_grad()  # Clear gradients
    loss = model(Y, X, mu_lambda_beta, sigmasq_lambda_beta, M_S_star, mu_W, eta_X_sq, lambda_a2, lambda_b2, tau_X=0.01, n_piX_sample=50)
    loss.backward()  # Compute gradients
    optimizer.step()  # Update parameters
    

    # Generate samples from the learned parameters
    M = 100  # Number of samples to generate
    Ps = []
    for _ in range(M):
        # Sample from the learned distribution
        log_mu_Ps = sinkhorn_logspace(model.MX.data, niters=10)
        log_sigmasq_Ps = torch.special.expit(model.VX.data) * (0.25 - 0.01) + 0.01
        sampled_P = torch.exp(log_mu_Ps) + torch.sqrt(log_sigmasq_Ps) * torch.randn_like(log_sigmasq_Ps)
        Ps.append(sampled_P.cpu().numpy())

    # Calculate num_correct
    num_correct = np.zeros(M)
    for m, P in enumerate(Ps):
        # Round doubly stochastic matrix P to the nearest permutation matrix
        rounded_P= round_to_perm(P)
        num_correct[m] = np.sum(rounded_P * perm_matrix_x.T.cpu().numpy())

    # Print the number of correct matches
    print(f"Average number of correct matches: {np.mean(num_correct)}")
    
    # Store loss and error
    losses.append(loss.item())

    # Print loss and error every step
    print(f"Step {step}, Loss: {loss.item()}, Error: {np.mean(num_correct)}")

    # Stopping rule
    # if step > 0 and abs(prev_loss - loss.item()) < 1e-4:
    #     print(f"Stopping early at step {step} due to minimal loss change.")
    #     break
    # prev_loss = loss.item()




# Plot the loss
plt.figure(figsize=(18, 5))

# Plot the loss over steps
plt.subplot(1, 3, 1)
plt.plot(losses, label="Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Loss over Steps")
plt.legend()

# Plot the true permutation matrix
plt.subplot(1, 3, 2)
sns.heatmap(perm_matrix_x.cpu().numpy(), annot=False, cmap="viridis")
plt.title("Heatmap of True Permutation Matrix")

# Plot the final permutation matrix
plt.subplot(1, 3, 3)
sns.heatmap(current_perm_matrix.cpu().numpy(), annot=False, cmap="viridis")
plt.title("Heatmap of Final Permutation Matrix")

plt.tight_layout()
plt.show()

Average number of correct matches: 1.19
Step 0, Loss: 13848.0771484375, Error: 1.19
Average number of correct matches: 1.18
Step 1, Loss: 13734.81640625, Error: 1.18
Average number of correct matches: 1.19
Step 2, Loss: 13634.986328125, Error: 1.19
Average number of correct matches: 1.18
Step 3, Loss: 13539.54296875, Error: 1.18
Average number of correct matches: 1.17
Step 4, Loss: 13405.822265625, Error: 1.17
Average number of correct matches: 1.19
Step 5, Loss: 13304.0966796875, Error: 1.19
Average number of correct matches: 1.19
Step 6, Loss: 13211.7548828125, Error: 1.19
Average number of correct matches: 1.19
Step 7, Loss: 13117.267578125, Error: 1.19
Average number of correct matches: 1.18
Step 8, Loss: 13016.919921875, Error: 1.18
Average number of correct matches: 1.16
Step 9, Loss: 12923.919921875, Error: 1.16
Average number of correct matches: 1.18
Step 10, Loss: 12829.3173828125, Error: 1.18
Average number of correct matches: 1.19
Step 11, Loss: 12748.93359375, Error: 1.19
A

KeyboardInterrupt: 

In [24]:
import numpy as np
from scipy.special import expit
from scipy.optimize import linear_sum_assignment

def sinkhorn_logspace_np(logP, niters=10):
    for _ in range(niters):
        logP = logP - logsumexp(logP, axis=0, keepdims=True)
        logP = logP - logsumexp(logP, axis=1, keepdims=True)
    return logP

def logsumexp(a, axis=None, keepdims=False):
    a_max = np.max(a, axis=axis, keepdims=True)
    res = np.log(np.sum(np.exp(a - a_max), axis=axis, keepdims=True)) + a_max
    if not keepdims:
        res = np.squeeze(res, axis=axis)
    return res

def round_to_perm(P):
    row, col = linear_sum_assignment(-P)
    perm = np.zeros_like(P)
    perm[row, col] = 1
    return perm

class VIpiX_numpy:
    def __init__(self, n_locations):
        self.n_locations = n_locations
        self.MX = np.log(1/n_locations) * np.ones((n_locations, n_locations))
        self.VX = -2 * np.ones((n_locations, n_locations))
        self.current_M_X_star = (1/n_locations) * np.ones((n_locations, n_locations))
        self.current_V_X_star = np.eye(n_locations)

    def forward(self, Y, X, mu_lambda_beta, sigmasq_lambda_beta, M_S_star, mu_W,
                eta_X_sq, lambda_a2, lambda_b2, tau_X=0.1, n_piX_sample=100, seed=100):

        np.random.seed(seed)
        log_MX_tilde = sinkhorn_logspace_np(self.MX, niters=10)
        MX_tilde = np.exp(log_MX_tilde)

        B = Y.shape[0]
        elbo = 0.0
        current_M_X_star = np.copy(self.current_M_X_star)
        current_V_X_star = np.copy(self.current_V_X_star)

        for _ in range(n_piX_sample):
            noise = np.random.randn(self.n_locations, self.n_locations)
            Phi = MX_tilde + np.sqrt(expit(self.VX)*(2-0.01) + 0.01) * noise
            round_Phi = round_to_perm(Phi - 0.95 * np.min(Phi))
            current_piX = tau_X * Phi + (1 - tau_X) * round_Phi
            current_M_X_star += current_piX
            current_V_X_star += current_piX.T @ current_piX

            term1 = 0.0
            for j in range(B):
                Y_j = Y[j]
                X_j = X[j]
                mu_Wj = mu_W[j]

                temp = current_piX @ X_j
                part1 = -2 * mu_lambda_beta * Y_j.dot(temp)
                part2 = (mu_lambda_beta**2 + sigmasq_lambda_beta) * temp.dot(temp)
                part3 = 2 * mu_lambda_beta * temp.T @ M_S_star @ mu_Wj

                term1 += part1 + part2 + part3

            coeff = -lambda_a2 / (2 * lambda_b2)
            total_term1 = coeff * term1

            x_mk_squared = current_piX**2
            x_mk_minus1_squared = (current_piX - 1)**2

            exponent1 = -x_mk_squared / (2 * eta_X_sq)
            exponent2 = -x_mk_minus1_squared / (2 * eta_X_sq)

            log_term = logsumexp(np.stack([exponent1, exponent2]), axis=0)
            total_log_term = np.sum(log_term)

            neg_log_tauX = self.n_locations ** 2 * np.log(tau_X)

            entropy_term = 0.5 * np.sum(np.log(expit(self.VX)*(2-0.01) + 0.01))

            elbo += total_term1 + total_log_term + neg_log_tauX + entropy_term

        elbo /= n_piX_sample
        self.current_M_X_star = current_M_X_star / n_piX_sample
        self.current_V_X_star = current_V_X_star / n_piX_sample
        return -elbo


In [25]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit
from copy import deepcopy

# Wrap the model class instance
model = VIpiX_numpy(n_locations=n_locations)

# Flatten the parameters
def pack_params(MX, VX):
    return np.concatenate([MX.flatten(), VX.flatten()])

def unpack_params(params, n):
    MX = params[:n * n].reshape(n, n)
    VX = params[n * n:].reshape(n, n)
    return MX, VX

# Objective function for optimization
def objective(params, Y, X, mu_lambda_beta, sigmasq_lambda_beta, M_S_star, mu_W, 
              eta_X_sq, lambda_a2, lambda_b2, tau_X, n_sample):
    
    MX, VX = unpack_params(params, model.n_locations)
    model.MX = MX
    model.VX = VX
    loss = model.forward(Y, X, mu_lambda_beta, sigmasq_lambda_beta, M_S_star, mu_W, 
                         eta_X_sq, lambda_a2, lambda_b2, tau_X, n_sample)
    return loss

# Initial parameter vector
initial_params = pack_params(model.MX, model.VX)

# Callback to monitor progress
prev_loss = None
perm_matrix_x_numpy = perm_matrix_x.T.detach().numpy()  # Replace with true permutation if known

def callback(params):
    global prev_loss
    MX, VX = unpack_params(params, model.n_locations)
    log_MX_tilde = sinkhorn_logspace_np(MX, niters=10)
    approx_perm = round_to_perm(np.exp(log_MX_tilde))
    err = np.linalg.norm(approx_perm - perm_matrix_x.T, ord='fro')
    variance = np.linalg.norm(expit(VX) * (2 - 0.01) + 0.01, ord='fro')
    
    current_loss = objective(params, Y.detach().numpy(), X.detach().numpy(), mu_lambda_beta, sigmasq_lambda_beta, M_S_star.detach().numpy(), 
                             mu_W.detach().numpy(),
                             eta_X_sq, lambda_a2, lambda_b2, 0.001, n_sample)
    
    print(f"Loss: {current_loss:.6f}, Error: {err:.6f}, Variance: {variance:.6f}")
    
    if prev_loss is not None and abs(prev_loss - current_loss) < 1e-6:
        raise StopIteration("Early stopping due to minimal loss change.")
    
    prev_loss = current_loss

# Run optimization
try:
    result = minimize(
        objective,
        initial_params,
        args = (
            Y.detach().numpy(),                     # shape: (B, d)
            X.detach().numpy(),                     # shape: (B, d)
            mu_lambda_beta,                  # scalar
            sigmasq_lambda_beta,            # scalar
            M_S_star.detach().numpy(),             # shape: (d, d)
            mu_W.detach().numpy(),                 # shape: (B, d)
            eta_X_sq,                        # scalar
            lambda_a2,                       # scalar
            lambda_b2,                       # scalar
            0.001,                                  # tau_X (float)
            50                                # n_piX_sample (int)
        ),
        method='L-BFGS-B',
        jac=None,  # No gradients; numerical estimation
        callback=callback,
        options={'maxiter': 200}
    )
except StopIteration as e:
    print(e)

# Final parameters
model.MX, model.VX = unpack_params(result.x, model.n_locations)


TypeError: unsupported operand type(s) for -: 'numpy.ndarray' and 'Tensor'

In [ ]:
# Initialize a tensor to accumulate the permutation matrices
perm_matrices_sum = torch.zeros_like(model.MX)

# Run the process 100 times
for _ in range(1000):
    # Calculate Phi
    log_MX_tilde_res = sinkhorn_logspace(model.MX, niters=10)
    MX_tilde_res = torch.exp(log_MX_tilde_res)
    Phi_res = MX_tilde_res + torch.sqrt(torch.exp(model.VX)) * torch.randn_like(model.VX)

    # Apply Hungarian algorithm on Phi
    #hungarian_result = hungarian_algorithm(-Phi)
    hungarian_result = torch.tensor(round_to_perm((Phi_res - 0.95*Phi_res.min()).detach().numpy()), dtype=torch.float)

    # Accumulate the permutation matrix
    perm_matrices_sum += hungarian_result

# Calculate the mean permutation matrix
mean_perm_matrix = perm_matrices_sum / 1000

# Print the mean permutation matrix
print("Mean Permutation Matrix:")
print(mean_perm_matrix)